# PICK: Semi-Supervised Segmentation for BHSD on Kaggle GPU

### Hướng dẫn thiết lập môi trường Kaggle:
1. **Notebook Settings (Bên phải màn hình):**
   - **Accelerator**: Chọn **GPU T4 x 2** hoặc **GPU P100** (16GB VRAM).
   - **Internet**: Bật sang **ON** (để cài đặt thư viện `medpy`, `tensorboardX`).
2. **Add Data (Góc trên bên phải):**
   - Bấm **+ Add Data** -> Chọn Dataset `preprocessed-bhsd-pick` mà bạn đã tải lên.

### Bước 1: Cài đặt các thư viện cần thiết

In [ ]:
!pip install -q medpy tensorboardX
!pip install -q h5py SimpleITK

### Bước 2: Thiết lập mã nguồn PICK
Chọn một trong hai cách dưới đây để đưa mã nguồn vào `/kaggle/working/PICK`.

In [ ]:
import os, glob

# Cách 1: Giải nén nếu bạn add pick_code.zip làm dataset
zip_files = glob.glob('/kaggle/input/**/pick_code.zip', recursive=True)
if zip_files:
    print(f"Tìm thấy file mã nguồn: {zip_files[0]}")
    !unzip -q {zip_files[0]} -d /kaggle/working/PICK
else:
    # Cách 2: Clone từ GitHub (nếu bạn đẩy lên repo cá nhân)
    if not os.path.exists('/kaggle/working/PICK'):
        # Thay bằng link repo của bạn nếu cần:
        # !git clone https://github.com/<your_username>/PICK.git /kaggle/working/PICK
        print("Chưa giải nén hoặc clone. Vui lòng tải mã nguồn vào thư mục /kaggle/working/PICK")

%cd /kaggle/working/PICK
!ls -la

### Bước 3: Kiểm tra GPU & Đường dẫn Dataset BHSD

In [ ]:
import torch, os, glob

print("=== KIỂM TRA GPU ===")
print("CUDA khả dụng:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Thiết bị:", torch.cuda.get_device_name(0))
    print("Bộ nhớ VRAM:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "GB")

print("\n=== KIỂM TRA DATASET ===")
# Tự động tìm đường dẫn chứa thư mục 'data' và 'splits'
candidates = glob.glob("/kaggle/input/**/data", recursive=True)
if candidates:
    DATA_ROOT = os.path.dirname(candidates[0])
    print(f"[OK] Đã tự động phát hiện DATA_ROOT: {DATA_ROOT}")
    print("Nội dung DATA_ROOT:", os.listdir(DATA_ROOT)[:5], "...")
else:
    # Mặc định theo tên dataset thông thường
    DATA_ROOT = "/kaggle/input/preprocessed-bhsd-pick"
    print(f"[!] Chưa tìm thấy tự động, gán mặc định DATA_ROOT: {DATA_ROOT}")

### Bước 4: Huấn luyện PICK (Binary hoặc Multi-class)

- Bạn có thể chọn Fold (0, 1, 2, 3, hoặc 4).
- Mỗi fold chạy 4,000 pre-train + 12,000 self-train iterations mất khoảng ~50-70 phút trên T4/P100.
- Kết quả models và logs được lưu vào `/kaggle/working/experiments`.

In [ ]:
# === HUẤN LUYỆN BINARY SEGMENTATION (FOLD 0) ===
FOLD_ID = 0

!python BHSD_binary_PICK_train.py \
    --root_path {DATA_ROOT} \
    --exp BHSD_binary_PICK_5folds \
    --exp_dir /kaggle/working/experiments \
    --fold {FOLD_ID} \
    --model VNet \
    --labelnum 15 \
    --max_samples 2200 \
    --batch_size 4 \
    --labeled_bs 2 \
    --gpu 0

### (Tùy chọn) Chạy Huấn Luyện Multi-class

In [ ]:
# === HUẤN LUYỆN MULTI-CLASS (FOLD 0) ===
# FOLD_ID = 0
# !python BHSD_multiclass_PICK_train.py \
#     --root_path {DATA_ROOT} \
#     --exp BHSD_multiclass_PICK_5folds \
#     --exp_dir /kaggle/working/experiments \
#     --fold {FOLD_ID} \
#     --model VNet \
#     --labelnum 15 \
#     --max_samples 2200 \
#     --batch_size 4 \
#     --labeled_bs 2 \
#     --gpu 0

### Bước 5: Nén và Xuất Kết quả Model / Log để Tải về
Sau khi chạy xong, file `.tar.gz` sẽ xuất hiện ở mục **Output** của Kaggle Notebook để bạn tải về máy tính chỉ với 1 click.

In [ ]:
!tar -czvf /kaggle/working/bhsd_pick_fold0_results.tar.gz -C /kaggle/working/experiments .
print("Hoàn tất nén! File đã sẵn sàng tại /kaggle/working/bhsd_pick_fold0_results.tar.gz")